# Conversation metrics demo

This demo records multi-turn conversations with a shared `conversation_id`, evaluates each conversation once, and persists the result for the TruLens dashboard.

In [ ]:
import os
from pathlib import Path

from trulens.apps.app import TruApp
from trulens.core import Metric
from trulens.core.otel.instrument import instrument
from trulens.core.session import TruSession
from trulens.otel.semconv.trace import SpanAttributes

os.environ["TRULENS_OTEL_TRACING"] = "1"
DB_PATH = Path.cwd() / "conversation_metrics_demo.sqlite"
if DB_PATH.exists():
    DB_PATH.unlink()

In [ ]:
def response_usage_attributes(ret, exception, *args, **kwargs):
    question = kwargs.get("question", args[-1] if args else "")
    prompt_tokens = 20 + len(str(question).split())
    completion_tokens = 8 + len(str(ret or "").split())
    total_tokens = prompt_tokens + completion_tokens
    cost = prompt_tokens * 0.000005 + completion_tokens * 0.000015
    return {
        SpanAttributes.COST.MODEL: "demo-support-model",
        SpanAttributes.COST.CURRENCY: "USD",
        SpanAttributes.COST.NUM_PROMPT_TOKENS: prompt_tokens,
        SpanAttributes.COST.NUM_COMPLETION_TOKENS: completion_tokens,
        SpanAttributes.COST.NUM_TOKENS: total_tokens,
        SpanAttributes.COST.COST: cost,
    }


class SupportAssistant:
    @instrument(attributes=response_usage_attributes)
    def respond(self, question: str) -> str:
        responses = {
            "How do I reset my password?": "Open Settings, choose Security, then select Reset password.",
            "Where is Security?": "Security is in the left navigation under Account.",
            "I do not see Reset password.": "Select Sign-in methods, then choose Password and Reset.",
            "Will that sign me out everywhere?": "Yes, resetting your password revokes your active sessions.",
            "Can I keep this browser signed in?": "No. For security, every session must authenticate again.",
            "What if I cannot access my email?": "Use the account recovery link and choose another verified method.",
            "Can an admin reset it for me?": "An admin can issue a temporary password if your policy allows it.",
            "How long is the temporary password valid?": "It expires after 24 hours or immediately after first use.",
            "Do I need MFA after the reset?": "Yes, your existing MFA requirement still applies.",
            "Can I change my MFA device too?": "After signing in, open Security and select Manage MFA devices.",
            "Is the password history enforced?": "Yes, your new password cannot match a recently used password.",
            "Can you summarize the steps?": "Reset the password, sign in again with MFA, then review active devices in Security.",
            "Can I export invoices?": "Yes. Open Billing, select Invoices, then choose Export CSV.",
        }
        return responses[question]


def conversation_completeness(
    records: list[dict],
) -> tuple[float, dict[str, str]]:
    answered_turns = sum(bool(record["output"]) for record in records)
    score = answered_turns / len(records)
    explanation = (
        f"The assistant answered {answered_turns} of {len(records)} turns. "
        "Every user question received a non-empty response."
    )
    return score, {"explanation": explanation}


session = TruSession(database_url=f"sqlite:///{DB_PATH}")
session.reset_database()
app = SupportAssistant()
metric = Metric(
    implementation=conversation_completeness,
    name="Conversation Completeness",
).on_conversation()
recorder = TruApp(
    app,
    app_name="Conversation Metrics Demo",
    app_version="v1",
    main_method=app.respond,
    feedbacks=[metric],
)

In [ ]:
password_reset_questions = [
    "How do I reset my password?",
    "Where is Security?",
    "I do not see Reset password.",
    "Will that sign me out everywhere?",
    "Can I keep this browser signed in?",
    "What if I cannot access my email?",
    "Can an admin reset it for me?",
    "How long is the temporary password valid?",
    "Do I need MFA after the reset?",
    "Can I change my MFA device too?",
    "Is the password history enforced?",
    "Can you summarize the steps?",
]

for question in password_reset_questions:
    with recorder(conversation_id="password-reset"):
        app.respond(question)

with recorder(conversation_id="invoice-export"):
    app.respond("Can I export invoices?")

session.force_flush()
recorder.compute_feedbacks()
session.force_flush()

In [ ]:
events = session.get_events(
    app_name="Conversation Metrics Demo",
    app_version="v1",
)
record_roots = events[
    events["record_attributes"].apply(
        lambda attributes: attributes.get(SpanAttributes.SPAN_TYPE)
        == SpanAttributes.SpanType.RECORD_ROOT
    )
]
eval_roots = events[
    events["record_attributes"].apply(
        lambda attributes: attributes.get(SpanAttributes.SPAN_TYPE)
        == SpanAttributes.SpanType.EVAL_ROOT
    )
]
usage_by_conversation = {}
for conversation_id in (
    record_roots["record_attributes"]
    .apply(lambda attributes: attributes.get(SpanAttributes.CONVERSATION_ID))
    .dropna()
    .unique()
):
    conversation_events = events[
        events["record_attributes"].apply(
            lambda attributes: attributes.get(SpanAttributes.CONVERSATION_ID)
            == conversation_id
        )
    ]
    usage_by_conversation[conversation_id] = {
        "total_tokens": sum(
            attributes.get(SpanAttributes.COST.NUM_TOKENS, 0)
            for attributes in conversation_events["record_attributes"]
        ),
        "total_cost_usd": sum(
            attributes.get(SpanAttributes.COST.COST, 0.0)
            for attributes in conversation_events["record_attributes"]
            if attributes.get(SpanAttributes.SPAN_TYPE)
            not in {
                SpanAttributes.SpanType.EVAL,
                SpanAttributes.SpanType.EVAL_ROOT,
            }
        ),
    }

results = [
    {
        "conversation_id": attributes[SpanAttributes.CONVERSATION_ID],
        "score": attributes[SpanAttributes.EVAL_ROOT.SCORE],
        **usage_by_conversation[attributes[SpanAttributes.CONVERSATION_ID]],
        "source_record_root_span_ids": attributes[
            f"{SpanAttributes.EVAL_ROOT.ARGS_SPAN_ID}.records"
        ],
    }
    for attributes in eval_roots["record_attributes"]
]
results

## Dashboard

Run the next cell to open the dashboard backed by this notebook's SQLite database. Stop it with `session.stop_dashboard()`.

In [ ]:
from trulens.dashboard import run_dashboard

dashboard = run_dashboard(session=session, force=True)
dashboard